# Calculate outcomes and admissions by LSOA

Outcomes assume that the whole population is treated.

__Results__:

For the base scenario and with adding MT to both Taunton and Worcester, and for including and excluding lossy drip-and-ship transfers, save these files:
+ `results_full_{scenario_name}.csv`
+ `results_stats_{scenario_name}.csv`
+ `results_stats_by_unit_{scenario_name}.csv`


## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.add_net_benefit import calculate_cohort_outcomes_admissions

In [2]:
dir_output = 'output'
file_geo_outcomes = 'results_geo_outcomes_'  # + scenario.csv

## Parameters

There must be a matching geo_outcomes file for each of these scenarios:

In [3]:
scenarios = ['base', 'taunworc']
scenarios = [f'{s}_bias10min' for s in scenarios]

Fixed proportion of patients have LVO (rest have nLVO), and options for proportion transferred for MT are gathered in the same results file.

In [4]:
prop_lvo = 0.3
options_prop_lvo_dripship_treated_with_mt = [1.0, 0.67]

# MT rate only affects admissions and transfers, not outcomes:
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

## Run for all scenarios: main changes

Save a copy of the full results to file.

For each set of changed units, calculate the new geography and outcomes data. Then for each set of stroke population parameters, combine the outcomes into the full population.

In [5]:
for scenario in scenarios:
    # Load data:
    file_geo_outcomes_here = file_geo_outcomes + f'{scenario}.csv'
    p = os.path.join(dir_output, file_geo_outcomes_here)
    df_geo_outcomes = pd.read_csv(p, index_col='LSOA')
    cols_start = df_geo_outcomes.columns

    results = calculate_cohort_outcomes_admissions(
        df_geo_outcomes,
        options_prop_lvo_dripship_treated_with_mt,
        col_admissions,
        mt_rate,
        prop_lvo,
    )
    cols_new = [c for c in results.columns if c not in cols_start]
    
    # Get descriptive stats for all numeric columns
    descriptive_stats = results.describe(percentiles=[0.25, 0.5, 0.75]).T
    descriptive_stats = descriptive_stats.round(5)
    
    # Repeat descriptive stats, grouped by nearest_ivt_unit
    results_stats_by_unit = results.groupby('nearest_ivt_unit').describe().T
    # Unpack index column so two-level index becomes two columns
    results_stats_by_unit = results_stats_by_unit.reset_index()
    rename_dict = {'level_0': 'variable', 'level_1': 'statistic'}
    results_stats_by_unit = results_stats_by_unit.rename(columns=rename_dict)
    results_stats_by_unit = results_stats_by_unit.round(5)
    
    # Save results:
    f = f'results_full_{scenario}.csv'
    results[cols_new].to_csv(os.path.join(dir_output, f))
    
    f = f'results_stats_{scenario}.csv'
    descriptive_stats.to_csv(os.path.join(dir_output, f))
    
    f = f'results_stats_by_unit_{scenario}.csv'
    results_stats_by_unit.to_csv(os.path.join(dir_output, f))

## Conclusion

Calculated full results, descriptive stats, and stats by unit and saved to file.